# Set WD to Root

In [1]:
from pathlib import Path
import os

repo_root = Path.cwd().parents[0]
os.chdir(repo_root)

# print(Path.cwd())

# Load Hyperparameters

In [7]:
# Load hyperparamters
import json

PARAM_PATH = Path('./config/test_rnn_params.json')

with open(PARAM_PATH, "r", encoding="utf-8") as f:
    params = json.load(f)

print(params)

{'batch_size': 200, 'learning_rate': 0.01, 'epochs': 5000, 'hidden_size': 100, 'embedding_dim': 100, 'bidirectional': True, 'architecture': 'gru', 'enable_attention': True}


# Load Data Loader

In [5]:
from scripts.data_loader.piqa_dataloader import get_piqa_dataloaders
from pathlib import Path
import sentencepiece as spm

BASE_PATH = Path('./datasets/PIQA')
BASE_PATH = BASE_PATH.resolve()

tokenizer = spm.SentencePieceProcessor(
    model_file="./scripts/tokenizer/piqa_bpe_v2.model"
)

train_loader, validate_loader, test_loader = get_piqa_dataloaders(base_path=BASE_PATH,
                                                                  tokenizer=tokenizer,
                                                                  batch_size=params['batch_size'])

In [12]:
# 提取一个 Batch 数据
input_1, input_2, mask_1, mask_2, labels = next(iter(train_loader))

# ----------------- 数据校验 -----------------
print("1. Label 形状与数值分布:")
print("Labels shape:", labels.shape)         # 应该输出: torch.Size([8])
print("Labels dtype:", labels.dtype)         # 应该输出: torch.int64
print("Labels unique values:", labels.unique()) # 必须只包含 tensor([0, 1])

print("\n2. Mask 逻辑校验:")
print("input_1[0]:", input_1[0])
print("mask_1[0]:", mask_1[0])
print("Pad ID:", tokenizer.pad_id())
assert tokenizer.pad_id() >= 0, "错误！SentencePiece 没有设置合法的 pad_id"

print("\n3. 文本拼接还原验证 (检查 CLS/SEP/EOS 是否正确嵌入):")
print("Seq 1 decoded:", tokenizer.decode(input_1[0].tolist()))
print("Seq 2 decoded:", tokenizer.decode(input_2[0].tolist()))

1. Label 形状与数值分布:
Labels shape: torch.Size([200])
Labels dtype: torch.int64
Labels unique values: tensor([0, 1])

2. Mask 逻辑校验:
input_1[0]: tensor([   4,  363,  328,  386, 1427,  263,  594, 2673, 4511,  278,  735,  306,
        1003,  501,  796, 7922,    5,  414,  374,  821,  832, 1898, 5787, 2008,
        7915,  504,  392,  282,  427,  278, 2762, 7913,    2,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,    3,
           3,    3,    3,    3,    3,    3,    3,    3,    3

# Load Model

In [16]:
from scripts.models.rnn_attention_refactored import RNNWithAttention
import torch.optim as optim
import torch.nn.functional as F

# 初始化模型和优化器
rnn_model = RNNWithAttention(vocab_size=tokenizer.vocab_size(),
                                 embedding_dim=params['embedding_dim'],
                                 hidden_dim=params['hidden_size'],
                                 padding_idx=tokenizer.pad_id(),
                                 architecture=params['architecture'],
                                 bidirectional=params['bidirectional'],
                                 enable_attention=params['enable_attention']
                                 )

rnn_optimiser = optim.Adam(params=rnn_model.parameters(), lr=params['learning_rate'])
criterion = F.cross_entropy

# 训练循环

In [17]:
from time import perf_counter

rnn_model.train()

epoch_loss = 0.0

for input_1, input_2, mask_1, mask_2, labels in train_loader:

    start_ts = perf_counter()
    # Forward
    rnn_optimiser.zero_grad()

    logits, weights = rnn_model.forward(input_1, input_2, mask_1, mask_2)

    # Loss
    loss = criterion(
        input=logits,
        target=labels
    )

    # Backward
    loss.backward()

    rnn_optimiser.step()

    # Record loss
    epoch_loss += loss.item()

    end_ts = perf_counter()
    time_elips = end_ts - start_ts
    
    print(
        f"time={time_elips:.2f}, "
        f"loss={loss.item():.2f}"
    )


time=3.09, loss=0.69
time=2.17, loss=0.69
time=2.53, loss=0.70
time=2.03, loss=0.69
time=2.64, loss=0.71
time=1.87, loss=0.69
time=2.34, loss=0.69
time=2.92, loss=0.69
time=8.45, loss=0.69
time=2.81, loss=0.70
time=4.66, loss=0.70
time=3.50, loss=0.70
time=3.02, loss=0.70
time=2.54, loss=0.70


KeyboardInterrupt: 